# Calibration

Walks through the per-frame radiometry chain: dark subtraction, flat-field correction, gain multiply, spectral response weighting, atmospheric correction.

The order is load-bearing -- flat-field has to be applied *before* the gain multiply, otherwise a non-uniform flat introduces a 1-3% bias that no unit test catches without explicit ordering checks.

In [ ]:
import numpy as np
from smokesight._sensor import SensorModel
from smokesight._atmos import IdentityAtmos

sensor = SensorModel.from_config({
    'sensor': {'gain': 0.012, 'bit_depth': 14, 'ner': 0.002}
}, frame_shape=(64, 64))

print(f'gain  = {sensor.gain} W m^-2 sr^-1 um^-1 per DN')
print(f'NER   = {sensor.noise_equivalent_radiance} W m^-2 sr^-1 um^-1')
print(f'flat shape: {sensor.flat.shape}, mean: {sensor.flat.mean():.3f}')
print(f'dark shape: {sensor.dark.shape}, mean: {sensor.dark.mean():.3f}')

### Step 1: dark subtraction

The dark frame captures the offset present even with no light. For a 14-bit sensor at room temp, dark might be 100-300 DN.

In [ ]:
raw_frame = np.full((64, 64), 5000.0, dtype=np.float32)
dark_frame = np.full_like(raw_frame, 100.0)
after_dark = raw_frame - dark_frame
print(f'after dark subtraction: mean DN = {after_dark.mean():.1f}')

### Step 2: flat-field divide

Per-pixel sensitivity variations are captured in a flat-field image. Dividing normalises each pixel's response to the array mean.

In [ ]:
flat = np.ones((64, 64), dtype=np.float32)
flat[10, 10] = 2.0  # one pixel is 2x more sensitive
flat = flat / flat.mean()  # mean = 1.0 (loader convention)

after_flat = after_dark / flat
print(f'corrected value at (10, 10): {after_flat[10, 10]:.1f} DN')
print(f'unchanged elsewhere:          {after_flat[20, 20]:.1f} DN')

### Step 3: gain to radiance

The gain was determined in lab calibration against a known light source. Multiplying converts DN to absolute radiance in W m^-2 sr^-1 um^-1.

In [ ]:
after_gain = after_flat * sensor.gain
print(f'radiance: {after_gain.mean():.4f} W m^-2 sr^-1 um^-1')

### Why the order matters

With a non-zero dark current, applying gain *before* dividing by the flat field gives a different answer than the correct order:

In [ ]:
wrong_order = raw_frame * sensor.gain - dark_frame * sensor.gain / flat
right_order = (raw_frame - dark_frame) / flat * sensor.gain

print(f'correct order, pixel (10, 10): {right_order[10, 10]:.4f}')
print(f'wrong order,   pixel (10, 10): {wrong_order[10, 10]:.4f}')
bias = (wrong_order[10, 10] - right_order[10, 10]) / right_order[10, 10] * 100
print(f'bias: {bias:.2f}%')

### Step 4: spectral response weighting

For multi-band sensors, each band has its own QE. SmokeSight broadcasts the per-pixel radiance over the wavelength axis.

In [ ]:
mb_sensor = SensorModel.from_config({
    'sensor': {
        'gain': 0.012, 'bit_depth': 14, 'ner': 0.002,
        'spectral_response': {
            'wavelengths': [3.5, 4.0, 4.5, 5.0],
            'response':    [0.3, 0.8, 0.9, 0.4],
        },
    }
}, frame_shape=(64, 64))

L_panchromatic = np.full((64, 64), 35.0, dtype=np.float32)
L_per_band = L_panchromatic[..., None] * mb_sensor.spectral_response
print(f'per-band radiance at (0, 0): {L_per_band[0, 0]}')
print(f'wavelengths [um]:            {mb_sensor.wavelengths}')

### Step 5: atmospheric correction

If py6s or pymodtran is configured, the package corrects for atmospheric transmittance and path radiance. For demonstration we use IdentityAtmos which leaves the radiance unchanged.

In [ ]:
atmos = IdentityAtmos()
L_surface = atmos.correct(L_per_band)
sigma_atmos = atmos.uncertainty(L_per_band)
print(f'identity correction unchanged: {np.allclose(L_surface, L_per_band)}')
print(f'atmospheric uncertainty:       {sigma_atmos.max():.4f}')